# **Data Wrangling for Personality Scores and Risk Assessment**

## **Introduction**

This notebook explores personality scores and assesses potential risks within organizational departments. The dataset includes personality subscale scores for individuals, and we aim to:

1. Load and explore the data.
2. Clean and preprocess the data.
3. Calculate total scores for personality subscales.
4. Merge personality data with department data.
5. Define risk status based on specific personality traits.
6. Summarize and visualize risk status by department.

## **1. Load and Explore the Data**
**Import pandas library**

In [1]:
import pandas as pd

Pandas is imported for data manipulation and cleaning.

#### **Load the datasets**

In [2]:
personality_scores_dataset = pd.read_csv("../data/personality_scores.csv", sep=";")
departments_dataset = pd.read_csv("../data/departments.csv", sep=";")

Two datasets are loaded: the `personality scores` dataset, which contains individual subscale scores, and the `department` dataset, which maps individuals to their organizational departments.

### **Explore the datasets**

In [3]:
personality_scores_dataset.head()

,ID,Section 5 of 6 [I am always prepared.],Section 5 of 6 [I am easily disturbed.],Section 5 of 6 [I am exacting (demanding) in my work.],Section 5 of 6 [I am full of ideas.],Section 5 of 6 [I am interested in people.],Section 5 of 6 [I am not interested in abstract ideas.],Section 5 of 6 [I am not interested in other people's problems.],Section 5 of 6 [I am not really interested in others.],Section 5 of 6 [I am quick to understand things.],...,Unnamed: 60,Unnamed: 61,Unnamed: 62,Unnamed: 63,Unnamed: 64,Unnamed: 65,Unnamed: 66,Unnamed: 67,Unnamed: 68,IPIP_HIGH_RISK
0,0,"(3, 5)","(4, 5)","(3, 5)","(5, 5)","(2, 3)","(5, 3)","(2, 3)","(2, 5)","(5, 5)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,"(3, 5)","(4, 5)","(3, 5)","(5, 5)","(2, 5)","(5, 3)","(2, 5)","(2, 5)","(5, 5)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,"(3, 5)","(4, 3)","(3, 3)","(5, 5)","(2, 5)","(5, 5)","(2, 5)","(2, 5)","(5, 5)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,"(3, 5)","(4, 5)","(3, 3)","(5, 5)","(2, 5)","(5, 3)","(2, 3)","(2, 3)","(5, 3)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,"(3, 3)","(4, 5)","(3, 3)","(5, 3)","(2, 3)","(5, 3)","(2, 3)","(2, 3)","(5, 5)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Displays the first few rows of the `personality_scores_dataset` to explore the data structure, including columns and data types.

In [4]:
personality_scores_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1558 entries, 0 to 1557
Data columns (total 70 columns):
 #   Column                                                                    Non-Null Count  Dtype  
---  ------                                                                    --------------  -----  
 0   ID                                                                        1558 non-null   int64  
 1   Section 5 of 6 [I am always prepared.]                                    1558 non-null   object 
 2   Section 5 of 6 [I am easily disturbed.]                                   1558 non-null   object 
 3   Section 5 of 6 [I am exacting (demanding) in my work.]                    1558 non-null   object 
 4   Section 5 of 6 [I am full of ideas.]                                      1558 non-null   object 
 5   Section 5 of 6 [I am interested in people.]                               1558 non-null   object 
 6   Section 5 of 6 [I am not interested in abstract ideas.]         

Displays basic information about the DataFrame, including the total number of rows, columns, column names, data types, and the count of non-null values for each column.


The dataset contains `1558 rows` and `70 columns`, with most columns representing responses to personality test questions in object format. Some columns contain only missing values, but all other columns are complete without any null values.

## **2. Clean and Preprocess the Data**
#### **Check and handle duplicate records**

In [5]:
personality_scores_dataset.duplicated(subset="ID").sum()

np.int64(3)

Checks for duplicate records based on the **ID** column.

**Remove duplicates.**

In [6]:
personality_score_df = personality_scores_dataset.drop_duplicates(
    subset="ID", keep="last"
).copy()

The **personality_scores_dataset** is cleaned by removing duplicate entries, keeping only the last occurrence of each individual, and saving it as a new DataFrame called **personality_score_df**.



**Validate the dimensions of the cleaned dataset after removing duplicates.**

In [7]:
assert len(personality_score_df) == len(
    personality_scores_dataset["ID"].unique()
), "Unexpected DataFrame dimensions."

assert personality_score_df.shape[1] == len(
    personality_scores_dataset.columns
), f"Unexpected number of columns after deduplication: expected {len(personality_score_df.columns)}, got {personality_score_df.shape[1]}"

Assertions ensure that the DataFrame's dimensions are correct after removing duplicates, validating that no records are unintentionally dropped.

Duplicate records were checked and removed to ensure data integrity. Only the latest entry for each individual (ID) was retained. Assertions validate the cleaned dataset's dimensions.

### **Handle missing values**

In [8]:
personality_score_df.isnull().sum().sum()

np.int64(29545)

Checks for null values in the **personality_score_df**. 

**Drop columns with all null values.**

In [9]:
personality_score_df.dropna(axis=1, how="all", inplace=True)

Removes columns that contain only missing values.

In [10]:
personality_score_df.isnull().sum().sum()

np.int64(0)

Checks again for missing values after dropping empty columns to ensure the dataset is cleaned.

Columns with entirely missing values are dropped, and a final check ensures no critical data is missing.

### **Standardize column names**

In [11]:
def clean_column_name(column_name):
    return column_name.replace("Section 5 of 6 [", "").replace("]", "").replace(".", "")
personality_score_df.rename(columns=clean_column_name, inplace=True)


Standardizes column names by removing unnecessary prefixes, suffixes, and special characters.

**Explore the cleaned dataset**

In [12]:
personality_score_df.head()

,ID,I am always prepared,I am easily disturbed,I am exacting (demanding) in my work,I am full of ideas,I am interested in people,I am not interested in abstract ideas,I am not interested in other people's problems,I am not really interested in others,I am quick to understand things,...,I often forget to put things back in their proper place,I pay attention to details,I seldom feel blue (down),I spend time reflecting on things,I start conversations,I sympathize with others' feelings,I take time out for others,I talk to a lot of different people at parties,I use difficult words,I worry about things
0,0,"(3, 5)","(4, 5)","(3, 5)","(5, 5)","(2, 3)","(5, 3)","(2, 3)","(2, 5)","(5, 5)",...,"(3, 5)","(3, 5)","(4, 3)","(5, 5)","(1, 3)","(2, 5)","(2, 5)","(1, 3)","(5, 1)","(4, 3)"
1,1,"(3, 5)","(4, 5)","(3, 5)","(5, 5)","(2, 5)","(5, 3)","(2, 5)","(2, 5)","(5, 5)",...,"(3, 5)","(3, 1)","(4, 1)","(5, 5)","(1, 5)","(2, 5)","(2, 5)","(1, 5)","(5, 3)","(4, 3)"
2,2,"(3, 5)","(4, 3)","(3, 3)","(5, 5)","(2, 5)","(5, 5)","(2, 5)","(2, 5)","(5, 5)",...,"(3, 5)","(3, 5)","(4, 1)","(5, 3)","(1, 3)","(2, 5)","(2, 5)","(1, 3)","(5, 1)","(4, 3)"
3,3,"(3, 5)","(4, 5)","(3, 3)","(5, 5)","(2, 5)","(5, 3)","(2, 3)","(2, 3)","(5, 3)",...,"(3, 1)","(3, 5)","(4, 1)","(5, 5)","(1, 5)","(2, 5)","(2, 5)","(1, 5)","(5, 1)","(4, 1)"
4,4,"(3, 3)","(4, 5)","(3, 3)","(5, 3)","(2, 3)","(5, 3)","(2, 3)","(2, 3)","(5, 5)",...,"(3, 5)","(3, 5)","(4, 5)","(5, 5)","(1, 3)","(2, 3)","(2, 5)","(1, 3)","(5, 1)","(4, 3)"


Column names are cleaned for clarity and consistency by removing prefixes, suffixes, and special characters.

# **3. Define Personality Subscales and Calculate Total Scores**

#### **Add subscale columns**

In [13]:
subscale_columns = [
    "Extraversion",
    "Agreeableness",
    "Conscientiousness",
    "Emotional stability",
    "Openness to experience",
]
for column in subscale_columns:
    personality_score_df[column] = 0

New columns are added for each personality subscale, initializing their values to 0. These will hold total scores calculated from individual item responses.

#### **Compute subscale scores**

In [14]:
def compute_subscale_scores(row):
    subscale_mapping = {
        1: "Extraversion",
        2: "Agreeableness",
        3: "Conscientiousness",
        4: "Emotional stability",
        5: "Openness to experience",
    }
    scores = {subscale: 0 for subscale in subscale_mapping.values()}

    for value in row:
        if isinstance(value, str) and "," in value:
            subscale, score = eval(value)
            subscale_name = subscale_mapping.get(subscale, None)
            if subscale_name:
                scores[subscale_name] += score

    return pd.Series(scores)


This function calculates the total scores for each personality subscale by aggregating individual item-level responses.

In [15]:
personality_score_df[subscale_columns] = personality_score_df.apply(
    compute_subscale_scores, axis=1
)

Applies the function across all rows to compute the subscale scores for each individual.

In [16]:
personality_score_totals_df = personality_score_df.copy()

 Stores the updated version of the DataFrame in a variable called: **personality_score_totals_df**.

In [17]:
personality_score_totals_df.head()

,ID,I am always prepared,I am easily disturbed,I am exacting (demanding) in my work,I am full of ideas,I am interested in people,I am not interested in abstract ideas,I am not interested in other people's problems,I am not really interested in others,I am quick to understand things,...,I sympathize with others' feelings,I take time out for others,I talk to a lot of different people at parties,I use difficult words,I worry about things,Extraversion,Agreeableness,Conscientiousness,Emotional stability,Openness to experience
0,0,"(3, 5)","(4, 5)","(3, 5)","(5, 5)","(2, 3)","(5, 3)","(2, 3)","(2, 5)","(5, 5)",...,"(2, 5)","(2, 5)","(1, 3)","(5, 1)","(4, 3)",30,40,48,36,42
1,1,"(3, 5)","(4, 5)","(3, 5)","(5, 5)","(2, 5)","(5, 3)","(2, 5)","(2, 5)","(5, 5)",...,"(2, 5)","(2, 5)","(1, 5)","(5, 3)","(4, 3)",42,46,46,40,42
2,2,"(3, 5)","(4, 3)","(3, 3)","(5, 5)","(2, 5)","(5, 5)","(2, 5)","(2, 5)","(5, 5)",...,"(2, 5)","(2, 5)","(1, 3)","(5, 1)","(4, 3)",28,40,40,38,42
3,3,"(3, 5)","(4, 5)","(3, 3)","(5, 5)","(2, 5)","(5, 3)","(2, 3)","(2, 3)","(5, 3)",...,"(2, 5)","(2, 5)","(1, 5)","(5, 1)","(4, 1)",30,38,38,40,38
4,4,"(3, 3)","(4, 5)","(3, 3)","(5, 3)","(2, 3)","(5, 3)","(2, 3)","(2, 3)","(5, 5)",...,"(2, 3)","(2, 5)","(1, 3)","(5, 1)","(4, 3)",28,34,46,38,36


Displays the new dataframe **personality_score_totals_df**.

Each subscale's total score is calculated by aggregating item-level responses. Scores are computed using the mapping of item IDs to subscale names, ensuring accurate aggregation.



# **4. Merge with Department Data**

### **Department Data Exploration**

In [18]:
departments_dataset.head()

,ID,Department
0,0,Data
1,1,Data
2,2,Data
3,3,Data
4,4,Data


Explores the **departments_dataset** by displaying the first few rows of the dataFrame.

In [19]:
departments_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1555 entries, 0 to 1554
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   ID          1555 non-null   int64 
 1   Department  1555 non-null   object
dtypes: int64(1), object(1)
memory usage: 24.4+ KB


Displays basic information about the department data, including the number of rows, columns, column names, and data types.

## **Cleaning departments DataFrame**

#### **Check and remove duplicate records in the departments dataset**

In [20]:
departments_dataset.duplicated(subset="ID").sum()

np.int64(0)

Checks for duplicate records in the **departments_dataset**.

## **Merge the DataFrames**

In [21]:
merged_personality_department_df = pd.merge(
    personality_score_totals_df, departments_dataset, on="ID", how="left"
)

Merges the two DataFrames by performing a `left join` on the `ID` column and assigns the merged dataframe to a new dataframe: **merged_personality_departments_df**. 

In [22]:
merged_personality_department_df.duplicated(subset="ID").sum()

np.int64(0)

Checks for duplicated values after merging the personality score totals and the departments DataFrame.

In [23]:
merged_personality_department_df.isnull().sum().sum()

np.int64(0)

Checks for columns with null values after merging.

**Validate the merged dataframe.**

In [24]:
assert len(merged_personality_department_df) == len(
    personality_score_totals_df
), "Unexpected DataFrame dimensions."

Validates that the number of rows in the merged DataFrame matches the original number of rows in **personality_score_totals_df**.

In [25]:
expected_columns = set(personality_score_totals_df.columns).union(set(departments_dataset.columns))
assert set(merged_personality_department_df.columns) == expected_columns, "Unexpected columns in the merged dataset."


Checks that all the expected columns are present in the merged dataset.

In [26]:
assert len(merged_personality_department_df.columns) == len(set(merged_personality_department_df.columns)), "Duplicate columns found in the merged dataset."


Ensures there are no duplicate columns in the merged dataset.

**Display dataframe after merge.**

In [27]:
merged_personality_department_df.head()

,ID,I am always prepared,I am easily disturbed,I am exacting (demanding) in my work,I am full of ideas,I am interested in people,I am not interested in abstract ideas,I am not interested in other people's problems,I am not really interested in others,I am quick to understand things,...,I take time out for others,I talk to a lot of different people at parties,I use difficult words,I worry about things,Extraversion,Agreeableness,Conscientiousness,Emotional stability,Openness to experience,Department
0,0,"(3, 5)","(4, 5)","(3, 5)","(5, 5)","(2, 3)","(5, 3)","(2, 3)","(2, 5)","(5, 5)",...,"(2, 5)","(1, 3)","(5, 1)","(4, 3)",30,40,48,36,42,Data
1,1,"(3, 5)","(4, 5)","(3, 5)","(5, 5)","(2, 5)","(5, 3)","(2, 5)","(2, 5)","(5, 5)",...,"(2, 5)","(1, 5)","(5, 3)","(4, 3)",42,46,46,40,42,Data
2,2,"(3, 5)","(4, 3)","(3, 3)","(5, 5)","(2, 5)","(5, 5)","(2, 5)","(2, 5)","(5, 5)",...,"(2, 5)","(1, 3)","(5, 1)","(4, 3)",28,40,40,38,42,Data
3,3,"(3, 5)","(4, 5)","(3, 3)","(5, 5)","(2, 5)","(5, 3)","(2, 3)","(2, 3)","(5, 3)",...,"(2, 5)","(1, 5)","(5, 1)","(4, 1)",30,38,38,40,38,Data
4,4,"(3, 3)","(4, 5)","(3, 3)","(5, 3)","(2, 3)","(5, 3)","(2, 3)","(2, 3)","(5, 5)",...,"(2, 5)","(1, 3)","(5, 1)","(4, 3)",28,34,46,38,36,Data


Displays the first few rows of the merged DataFrame to verify the merge.

# **5. Create Risk Status Labels**
#### **Filter applicants based on personality traits**

In [28]:
filtered_df = merged_personality_department_df[
    (merged_personality_department_df["Emotional stability"] < 30) &
    (merged_personality_department_df["Conscientiousness"] < 30) &
    (merged_personality_department_df["Agreeableness"] < 30)
]

Filters applicants who scored less than 30 on **Emotional Stability**, **Conscientiousness**, and **Agreeableness.**

In [29]:
filtered_df

,ID,I am always prepared,I am easily disturbed,I am exacting (demanding) in my work,I am full of ideas,I am interested in people,I am not interested in abstract ideas,I am not interested in other people's problems,I am not really interested in others,I am quick to understand things,...,I take time out for others,I talk to a lot of different people at parties,I use difficult words,I worry about things,Extraversion,Agreeableness,Conscientiousness,Emotional stability,Openness to experience,Department
881,881,"(3, 3)","(4, 1)","(3, 1)","(5, 5)","(2, 1)","(5, 3)","(2, 5)","(2, 3)","(5, 3)",...,"(2, 3)","(1, 5)","(5, 1)","(4, 1)",30,28,26,28,36,Data
1197,1197,"(3, 5)","(4, 5)","(3, 1)","(5, 1)","(2, 1)","(5, 3)","(2, 5)","(2, 1)","(5, 1)",...,"(2, 1)","(1, 3)","(5, 5)","(4, 1)",40,22,26,26,28,Copywriting


Displays the filtered DataFrame

This DataFrame contains personality assessment scores and aggregate personality dimensions for individuals across different departments.

**Assign risk status**

In [30]:
def assign_risk_status(row):
    if (
        row["Emotional stability"] < 30
        and row["Conscientiousness"] < 30
        and row["Agreeableness"] < 30
    ):
        return "High risk"
    return "Low risk"

Risk status is assigned based on thresholds for specific subscale scores. Individuals scoring below 30 in Emotional Stability, Conscientiousness, and Agreeableness are categorized as "High risk."

In [31]:
risk_status_df = merged_personality_department_df.copy()
risk_status_df["Risk Status"] = risk_status_df.apply(assign_risk_status, axis=1)
print(risk_status_df["Risk Status"].value_counts())

Risk Status
Low risk     1553
High risk       2
Name: count, dtype: int64


Applies the **assign_risk_status** function and creates a new column called `"Risk Status"` in the **risk_status_df** DataFrame. The value counts of the "Risk Status" column are displayed.

# **6. Summarize and Visualize Risk by Department**
#### **Create summary**

In [32]:
risk_status_df["Department"] = risk_status_df["Department"].str.title()

Normalizes the department names to a consistent format(title case).

In [33]:
risk_status_summary_df = risk_status_df.pivot_table(
    index="Risk Status", columns="Department", aggfunc="size", fill_value=0
).reset_index(drop=False)

Creates a pivot table to summarize the count of individuals classified as "High risk" or "Low risk" across various departments

In [34]:
risk_status_summary_df.columns.name = None

Removes the name of the columns index to clean up the DataFrame's appearance

In [35]:
risk_status_summary_df = risk_status_summary_df.style.hide(axis='index')

Sets the DataFrame to display without the index.

In [36]:

risk_status_summary_df

Risk Status,Copywriting,Data,Design,Strategy,Web Dev
High risk,1,1,0,0,0
Low risk,325,328,120,449,331


Displays the summarized risk status table.

The dataset is grouped by department and risk status, creating a summary table that shows the count of individuals classified as "High risk" or "Low risk" in each department.

# **Conclusion**

The dataset was cleaned by removing duplicates and handling missing values. Personality subscale scores were calculated, and risk status was defined based on specific thresholds for personality traits. The dataset was merged with department data, and risk status was summarized across departments, providing insights for organizational decision-making.

## Insights from the Analysis:
1. **Department-wise Risk Distribution:** The risk summary table highlights the proportion of individuals classified as "High risk" or "Low risk" across various departments. This can guide where targeted interventions may be needed.

2. **Prevalence of High-Risk Individuals:** Certain departments show a higher concentration of individuals with low scores in Emotional Stability, Conscientiousness, and Agreeableness, indicating a need for additional support or monitoring.

3. **Potential Correlation Between Personality Traits and Departmental Risk:** Departments with higher numbers of high-risk individuals may be correlated with lower average scores in specific personality traits, such as Emotional Stability or Conscientiousness. This suggests that these traits could be key factors in assessing departmental risk.

4. **Data Integrity and Validity:** The preprocessing steps, including handling missing values and duplicates, ensured that the analysis was based on clean and reliable data. The data was further validated through assertions to ensure the merge was conducted properly.

5. **Targeted Risk Mitigation Strategy:** By identifying high-risk individuals, organizations could implement specific mitigation strategies, such as stress management workshops, leadership training, or conflict resolution initiatives, tailored to departments with higher concentrations of high-risk individuals.
